# Which diagnoses travel together — NIS 2019

Every US hospital stay is tagged with a list of ICD-10-CM diagnosis codes.
This notebook asks which pairs of codes land on the same discharge record far
more often than chance would predict, across all **7,083,805 discharges** in
the HCUP National Inpatient Sample 2019.

The hard part is not finding associations — it is telling a real clinical
relationship apart from the several things that imitate one:

1. codes that ICD-10-CM *requires* to be recorded together,
2. two codes that are the same condition at different levels of detail,
3. two conditions that are simply both common in the same kind of patient,
4. a documentation habit at a handful of hospitals.

Each gets its own check below.

Prerequisite: `python src/build_transactions.py`.

In [ ]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd

from dataset import Dataset
import mine_associations as ma

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 55)

nis = Dataset()
n = len(nis)
print(f"{n:,} discharges | {len(nis.codes):,} distinct ICD-10-CM codes")
print(f"{nis.X.nnz:,} coded diagnoses, {nis.X.nnz / n:.1f} per discharge on average")

## 1. Count every pair, exactly

The data is held as a sparse discharge × code matrix. Multiplying it by its
own transpose (`XᵀX`) gives exact co-occurrence counts for every pair of
codes at once — about a minute for the full file. The dense equivalent would
be 7.08 M × 38,927 and would not fit in memory, which is the usual reason
this kind of analysis gets run on a sample instead.

Codes are restricted to those appearing on at least 3,500 discharges. That
keeps the pair matrix tractable while still covering the overwhelming
majority of coded diagnoses.

In [ ]:
counts = nis.counts
keep = np.flatnonzero(counts >= ma.MIN_CODE_COUNT)
codes_kept = nis.codes[keep]

share = 100 * counts[keep].sum() / counts.sum()
print(f"{len(keep):,} codes at >= {ma.MIN_CODE_COUNT:,} discharges "
      f"({share:.1f}% of all coded diagnoses)")

Xf, co = ma.pair_counts(nis.X, keep)
singles = np.asarray(Xf.sum(axis=0)).ravel()
print(f"{co.nnz // 2:,} co-occurring pairs observed")

## 2. Score them, and test them

For every pair we compute support, lift, both directional confidences,
leverage, Jaccard, and an odds ratio with a confidence interval.

Two choices matter here:

**Lift and odds ratio, not confidence.** Confidence is `P(B|A)` and has no
baseline term, so it scores ~1.0 whenever `B` is near-universal, whatever `A`
is. Lift and the odds ratio both compare the observed count against what
independence would predict.

**A multiplicity correction.** Hundreds of thousands of pairs are tested at
once. At the usual 5% threshold that manufactures thousands of false
positives on its own, so every interval is Bonferroni-corrected.

In [ ]:
pairs = ma.build_table(co, singles, n, codes_kept)
print(f"{len(pairs):,} pairs seen together at least {ma.MIN_PAIR_COUNT} times")
print(f"Bonferroni critical z = {pairs.z_crit.iloc[0]:.2f}")

pairs = ma.annotate(pairs, nis.lookup)
pairs = ma.suppress(pairs)   # HCUP rule: nothing from <= 10 discharges

sig = pairs[(pairs.or_ci_low > 1) & (pairs.lift > 1)].copy()
print(f"{len(sig):,} pairs significantly positive after correction")

## 3. Set aside the pairs that are true by construction

Several whole families of ICD-10-CM codes exist only to be recorded next to
another code, so finding them together is circular. Each is detected from the
code's own description rather than a hand-written list:

- **same CCSR category** — AHRQ groups ~73,000 ICD-10 codes into ~530
  clinical concepts. Two codes in one category are one condition at two
  levels of detail.
- **same 3-character root** — `E11.22` and `E11.65` are both type 2 diabetes.
- **manifestation wrappers** — "… in diseases classified elsewhere" cannot be
  coded alone; pairing it with its cause is the rule, not a finding.
- **obstetric wrappers** — "… complicating childbirth" is the pregnancy
  version of a condition already coded elsewhere on the record.
- **adverse-effect pairs** — one documented drug reaction always generates a
  drug code and an effect code.
- **restated descriptions** — "Sepsis due to Streptococcus pneumoniae" and
  "Pneumonia due to Streptococcus pneumoniae" are not a discovery.

In [ ]:
flags = ["same_ccsr", "same_icd3", "definitional", "context_code",
         "wrapper_code", "adverse_effect_pair", "same_concept_text"]
breakdown = pd.Series({f: int(sig[f].sum()) for f in flags})
breakdown["--> survive every filter"] = int((~sig.structural).sum())
breakdown.to_frame("pairs flagged")

In [ ]:
novel = sig[~sig.structural].copy()
novel.nlargest(15, "lift")[
    ["code_a", "desc_a", "code_b", "desc_b", "n_both", "lift", "odds_ratio"]
]

## 4. Adjust for who the patient is

Two conditions that are both common in 85-year-olds co-occur strongly without
being connected at all. Stratifying on age band and sex, then pooling the
strata with Mantel-Haenszel, removes that effect.

`confounding_ratio` is crude ÷ adjusted: near 1 means age and sex explained
none of the association, well above 1 means they explained most of it. Below
1 — which happens — means the crude figure *understated* the link.

In [ ]:
age = nis.meta["AGE"].to_numpy()
band = np.digitize(age, [1, 18, 45, 65, 75, 85])
sex = np.nan_to_num(nis.meta["FEMALE"].to_numpy(), nan=-1).astype(int)
strata = band * 10 + sex

col_of = {c: k for k, c in enumerate(codes_kept)}
top = novel[novel.n_both >= 500].nlargest(6000, "lift")
adj = ma.mantel_haenszel(Xf, col_of, list(zip(top.code_a, top.code_b)), strata)

novel = novel.merge(adj, on=["code_a", "code_b"], how="left")
novel["confounding_ratio"] = novel.odds_ratio / novel.or_adjusted
print(f"adjusted {adj.or_adjusted.notna().sum():,} candidate pairs")

In [ ]:
cols = ["code_a", "desc_a", "code_b", "desc_b", "n_both",
        "lift", "odds_ratio", "or_adjusted", "confounding_ratio"]
ranked = novel.dropna(subset=["or_adjusted"])
ranked[~ranked.obstetric].nlargest(20, "lift")[cols]

## 5. Ask where the data came from

This is the check that nothing statistical can substitute for.

A hospital that copies a patient's whole outpatient problem list onto the
inpatient record will produce a set of unrelated chronic complaints that
appear together over and over. That looks *identical* to a clinical
relationship in every metric above — same large odds ratio, same tight
interval, same lack of age confounding.

The only way to separate them is to count contributing hospitals. Run
`src/hospital_concentration.py`, which scores every surviving candidate and
calibrates the threshold against six relationships nobody disputes.

In [ ]:
import hospital_concentration as hc

calib = pd.concat([
    hc.concentration(hc.CONTROLS).assign(group="established clinical link"),
    hc.concentration(hc.SUSPECTS).assign(group="suspected documentation habit"),
], ignore_index=True)

calib[["group", "desc_a", "desc_b", "n_both",
       "hospitals_contributing", "concentration"]]

The two groups separate by roughly an order of magnitude on both measures.
Established links appear at hundreds to thousands of hospitals; the suspect
ones at 11 to 55 out of 4,568.

Note that concentration alone is not sufficient — cystic fibrosis scores high
because it is managed at accredited centres. It is the *number of
contributing hospitals* that distinguishes a specialist referral pattern from
a local documentation habit.

## 6. Save

Both full tables are aggregate and cell-suppressed, but together they run to
~160 MB, so they go to the gitignored `cache/`. `results/` keeps only what is
small enough to read and to publish.

Then `src/triage_findings.py` collapses near-duplicate code pairs into one
row per clinical concept, and `README.md` walks through what survived.

In [ ]:
from config import CACHE, RESULTS

sig.sort_values("lift", ascending=False).to_csv(
    CACHE / "all_significant_pairs.csv", index=False)
novel.sort_values("lift", ascending=False).to_csv(
    CACHE / "cross_category_pairs.csv", index=False)
pairs.sort_values("n_both", ascending=False).head(500).to_csv(
    RESULTS / "most_common_pairs.csv", index=False)
print(f"full tables -> {CACHE}\ntop 500 pairs -> {RESULTS}")